In [1]:

import os
import json
import re
import numpy as np
import pandas as pd

BASE_PATH = r"C:\Users\Neda\Desktop\personality_llm"
PROCESSED_DATA_PATH = os.path.join(BASE_PATH, "data", "processed")
FEATURES_PATH = os.path.join(BASE_PATH, "data", "features")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

os.makedirs(FEATURES_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

KAMTERA_CLEAN_PATH = os.path.join(PROCESSED_DATA_PATH, "kamtera_clean.csv")
TRAIN_FEATURES_PATH = os.path.join(FEATURES_PATH, "train_features.csv")

LLM_INPUT_PATH = os.path.join(PROCESSED_DATA_PATH, "kamtera_for_llm_labeling.csv")
LLM_OUTPUT_PATH = os.path.join(FEATURES_PATH, "llm_personality_features.csv")

print("Paths ready")


Paths ready


In [2]:

# =========================================
# 1. Load Kamtera clean data
# =========================================

kamtera_df = pd.read_csv(KAMTERA_CLEAN_PATH)
print(kamtera_df.shape)
print(kamtera_df.head())

TEXT_COL_CANDIDATES = ["clean_text", "text", "message", "content", "sentence"]
text_col = None
for c in TEXT_COL_CANDIDATES:
    if c in kamtera_df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(f"No text column found. Available columns: {kamtera_df.columns.tolist()}")

kamtera_df = kamtera_df[[text_col]].dropna().drop_duplicates().rename(columns={text_col: "clean_text"})
kamtera_df["clean_text"] = kamtera_df["clean_text"].astype(str).str.strip()
kamtera_df = kamtera_df[kamtera_df["clean_text"].str.len() > 10].reset_index(drop=True)

print("Clean Kamtera rows:", len(kamtera_df))
kamtera_df.head()


(107280, 11)
                                                text  \
0  مدت زمان بررسی توسط دادگاه بدوی پس از رای دادگ...   
1                                ادعای طلب بدون مدرک   
2                     تغییر پست کارکنان رسمی آزمایشی   
3                     اختلاف راه شراکتی بین دو مزرعه   
4                      شرایط قانونی اوراقی خودرو چیه   

                                          clean_text  num_words  num_chars  \
0  مدت زمان بررسی توسط دادگاه بدوی پس از رای دادگ...       12.0       58.0   
1                                ادعای طلب بدون مدرک        4.0       19.0   
2                     تغییر پست کارکنان رسمی آزمایشی        5.0       30.0   
3                     اختلاف راه شراکتی بین دو مزرعه        6.0       30.0   
4                      شرایط قانونی اوراقی خودرو چیه        5.0       29.0   

   avg_word_length  type_token_ratio  num_questions  num_exclamations  \
0         3.916667          0.916667            0.0               0.0   
1         4.000000          1.00000

,clean_text
0,مدت زمان بررسی توسط دادگاه بدوی پس از رای دادگ...
1,ادعای طلب بدون مدرک
2,تغییر پست کارکنان رسمی آزمایشی
3,اختلاف راه شراکتی بین دو مزرعه
4,شرایط قانونی اوراقی خودرو چیه


In [3]:

# =========================================
# 2. Sample texts for LLM labeling
# =========================================

N_SAMPLE = 300  # می‌توانی برای پایان‌نامه بیشترش کنی، مثلا 1000 یا 2000

sample_df = kamtera_df.sample(
    n=min(N_SAMPLE, len(kamtera_df)),
    random_state=42
).reset_index(drop=True)

sample_df.to_csv(LLM_INPUT_PATH, index=False, encoding="utf-8-sig")
print("Saved for LLM labeling:", LLM_INPUT_PATH)
sample_df.head()


Saved for LLM labeling: C:\Users\Neda\Desktop\personality_llm\data\processed\kamtera_for_llm_labeling.csv


,clean_text
0,طلاق به علت اعتیاد
1,استخدام فرزندان ایثارگران بدون کارت پایان خدمت...
2,نحوه محاسبه تجاری زمین کارگاه
3,چه راه کاری هست که وقت دادگاه جلو بندازیم
4,منفک شدن از خدمت ناجا


In [4]:

# =========================================
# 3. Prompt template for LLM
# =========================================

TRAITS = [
    "openness",
    "conscientiousness",
    "extraversion",
    "agreeableness",
    "neuroticism",
]

SYSTEM_PROMPT = """
You are a careful psychological text annotation assistant.
Your task is NOT clinical diagnosis.
Estimate Big Five personality signals only from the writing style and content.
Return valid JSON only.
Scores must be integers from 1 to 5.
confidence must be a float from 0 to 1.
""".strip()

USER_PROMPT_TEMPLATE = """
متن فارسی زیر را از نظر نشانه‌های زبانی مرتبط با پنج عامل بزرگ شخصیت تحلیل کن.
تشخیص تو فقط باید بر اساس همین متن باشد، نه حدس‌های خارج از متن.

Text:
{text}

Return JSON only with this exact schema:
{{
  "openness": 1-5,
  "conscientiousness": 1-5,
  "extraversion": 1-5,
  "agreeableness": 1-5,
  "neuroticism": 1-5,
  "confidence": 0.0-1.0
}}
""".strip()

print(USER_PROMPT_TEMPLATE.format(text="من دوست دارم چیزهای جدید را امتحان کنم."))


متن فارسی زیر را از نظر نشانه‌های زبانی مرتبط با پنج عامل بزرگ شخصیت تحلیل کن.
تشخیص تو فقط باید بر اساس همین متن باشد، نه حدس‌های خارج از متن.

Text:
من دوست دارم چیزهای جدید را امتحان کنم.

Return JSON only with this exact schema:
{
  "openness": 1-5,
  "conscientiousness": 1-5,
  "extraversion": 1-5,
  "agreeableness": 1-5,
  "neuroticism": 1-5,
  "confidence": 0.0-1.0
}


In [5]:

# =========================================
# 4. JSON parsing helper
# =========================================

def parse_llm_json(raw_text):
    """Extract and validate JSON from an LLM response."""
    if raw_text is None:
        return None
    raw_text = str(raw_text).strip()

    # remove code fences if present
    raw_text = re.sub(r"^```json", "", raw_text, flags=re.IGNORECASE).strip()
    raw_text = re.sub(r"^```", "", raw_text).strip()
    raw_text = re.sub(r"```$", "", raw_text).strip()

    # extract first JSON object
    match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)
    if match:
        raw_text = match.group(0)

    try:
        data = json.loads(raw_text)
    except Exception:
        return None

    out = {}
    for t in TRAITS:
        try:
            v = int(round(float(data.get(t, np.nan))))
            out[t] = int(np.clip(v, 1, 5))
        except Exception:
            out[t] = np.nan

    try:
        out["llm_confidence"] = float(data.get("confidence", data.get("llm_confidence", np.nan)))
        out["llm_confidence"] = float(np.clip(out["llm_confidence"], 0, 1))
    except Exception:
        out["llm_confidence"] = np.nan

    return out


In [6]:

# =========================================
# 5A. Option A: Fill labels manually / externally
# =========================================
# اگر با ChatGPT / Qwen / Llama به صورت بیرونی برچسب زدی، خروجی را با ستون‌های زیر ذخیره کن:
# clean_text, openness, conscientiousness, extraversion, agreeableness, neuroticism, llm_confidence

expected_columns = ["clean_text"] + TRAITS + ["llm_confidence"]
print("Expected output columns:")
print(expected_columns)
print("Save final file here:", LLM_OUTPUT_PATH)


Expected output columns:
['clean_text', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'llm_confidence']
Save final file here: C:\Users\Neda\Desktop\personality_llm\data\features\llm_personality_features.csv


In [9]:

# =========================================
# 5B. Optional: If you use an API, place your labeling code here
# =========================================
# نکته: این سلول عمداً اجرا نمی‌شود. برای اجرای واقعی، کد API یا مدل محلی خودت را جایگزین کن.

RUN_API_LABELING = False

if RUN_API_LABELING:
    rows = []
    for i, text in enumerate(sample_df["clean_text"].tolist()):
        prompt = USER_PROMPT_TEMPLATE.format(text=text)
        
        # response_text = call_your_llm(system=SYSTEM_PROMPT, user=prompt)
        response_text = None  # TODO: replace with actual LLM response
        parsed = parse_llm_json(response_text)
        
        if parsed is None:
            parsed = {t: np.nan for t in TRAITS}
            parsed["llm_confidence"] = np.nan
        
        parsed["clean_text"] = text
        rows.append(parsed)

    llm_features_df = pd.DataFrame(rows)[expected_columns]
    llm_features_df.to_csv(LLM_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print("Saved:", LLM_OUTPUT_PATH)


In [10]:

# =========================================
# 6. Fallback LLM-like weak features for pipeline testing only
# =========================================
# این بخش فقط برای تست اجرای pipeline است، نه نتیجه نهایی پایان‌نامه.
# برای دفاع، بهتر است این فایل را با خروجی واقعی LLM جایگزین کنی.

CREATE_FALLBACK_FOR_TEST = True

if CREATE_FALLBACK_FOR_TEST and not os.path.exists(LLM_OUTPUT_PATH):
    tmp = sample_df.copy()
    rng = np.random.default_rng(42)
    for t in TRAITS:
        tmp[t] = rng.integers(1, 6, size=len(tmp))
    tmp["llm_confidence"] = 0.50
    tmp = tmp[expected_columns]
    tmp.to_csv(LLM_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print("Fallback file created for code testing only:", LLM_OUTPUT_PATH)
else:
    print("LLM output already exists or fallback disabled.")


Fallback file created for code testing only: C:\Users\Neda\Desktop\personality_llm\data\features\llm_personality_features.csv
